# Modelo de Regresion Supervisada - Prediccion de Tiempo de Entrega

Esta es la linea de trabajo de Cesar para estimar el tiempo de entrega en Logistica 4.0. La pregunta del modelo es: **Cuanto demorara una entrega?**

## 1. Importacion de librerias y configuracion de rutas

In [ ]:
from pathlib import Path
import sys
import importlib

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_PATH = PROJECT_ROOT / "src"

if str(SRC_PATH) not in sys.path:
    sys.path.append(str(SRC_PATH))

DATA_PATH = PROJECT_ROOT / "data" / "raw" / "5_logistica_40.csv"
PROCESSED_DATA_PATH = PROJECT_ROOT / "data" / "processed" / "cesar_logistica_clean.csv"
BASELINE_PREDICTIONS_PATH = (
    PROJECT_ROOT / "results" / "reports" / "cesar_regression_baseline_predictions.csv"
)

METRICS_OUTPUT_PATH = PROJECT_ROOT / "results" / "metrics" / "cesar_regression_metrics.csv"
BEST_MODEL_SUMMARY_PATH = PROJECT_ROOT / "results" / "metrics" / "cesar_regression_best_model_summary.csv"

RMSE_PLOT_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_regression_rmse_comparison.png"
MAE_PLOT_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_regression_mae_comparison.png"
REAL_VS_PRED_PLOT_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_real_vs_predicho_best_model.png"
RESIDUALS_PLOT_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_residuals_best_model.png"

TUNING_RESULTS_PATH = PROJECT_ROOT / "results" / "metrics" / "cesar_regression_tuning_results.csv"
OPTIMIZED_METRICS_PATH = PROJECT_ROOT / "results" / "metrics" / "cesar_regression_optimized_metrics.csv"
BASE_VS_OPTIMIZED_PATH = PROJECT_ROOT / "results" / "metrics" / "cesar_regression_base_vs_optimized.csv"

BASE_VS_OPT_RMSE_PLOT_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_base_vs_optimized_rmse.png"
BASE_VS_OPT_MAE_PLOT_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_base_vs_optimized_mae.png"
OPTIMIZED_REAL_VS_PRED_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_optimized_real_vs_predicho.png"
OPTIMIZED_RESIDUALS_PATH = PROJECT_ROOT / "results" / "plots" / "cesar_optimized_residuals.png"

FINAL_MODEL_PATH = PROJECT_ROOT / "models" / "trained_models" / "cesar_regression_model.joblib"
FINAL_METRICS_PATH = PROJECT_ROOT / "results" / "metrics" / "cesar_regression_final_metrics.csv"
FINAL_SUMMARY_PATH = PROJECT_ROOT / "results" / "reports" / "cesar_regression_final_summary.md"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_PATH:", DATA_PATH)
print("PROCESSED_DATA_PATH:", PROCESSED_DATA_PATH)
print("BASELINE_PREDICTIONS_PATH:", BASELINE_PREDICTIONS_PATH)
print("METRICS_OUTPUT_PATH:", METRICS_OUTPUT_PATH)
print("BEST_MODEL_SUMMARY_PATH:", BEST_MODEL_SUMMARY_PATH)
print("TUNING_RESULTS_PATH:", TUNING_RESULTS_PATH)
print("OPTIMIZED_METRICS_PATH:", OPTIMIZED_METRICS_PATH)
print("BASE_VS_OPTIMIZED_PATH:", BASE_VS_OPTIMIZED_PATH)
print("FINAL_MODEL_PATH:", FINAL_MODEL_PATH)
print("FINAL_METRICS_PATH:", FINAL_METRICS_PATH)
print("FINAL_SUMMARY_PATH:", FINAL_SUMMARY_PATH)


## 2. Importacion de modulos propios

In [ ]:
import regression_setup
import data_preprocessing
import model_training
import model_evaluation
import hyperparameter_tuning
import model_persistence

importlib.reload(regression_setup)
importlib.reload(data_preprocessing)
importlib.reload(model_training)
importlib.reload(model_evaluation)
importlib.reload(hyperparameter_tuning)
importlib.reload(model_persistence)

from regression_setup import (
    TARGET_COLUMN,
    FEATURE_COLUMNS,
    NUMERIC_FEATURES,
    CATEGORICAL_FEATURES,
    load_dataset,
    split_features_target,
)

from data_preprocessing import (
    preprocess_regression_dataframe,
    get_missing_values_report,
    get_duplicate_report,
    build_regression_preprocessor,
    save_processed_dataset,
)

from model_training import (
    split_regression_data,
    build_regression_pipelines,
    train_regression_models,
    generate_baseline_predictions,
    save_baseline_predictions,
    run_regression_training_workflow,
)

from model_evaluation import (
    evaluate_regression_models,
    get_best_regression_model_name,
    create_predictions_dataframe,
    save_metrics,
    save_best_model_summary,
    plot_metric_comparison,
    plot_real_vs_predicted,
    plot_residuals,
    run_regression_evaluation_workflow,
)

from hyperparameter_tuning import (
    get_param_grid_for_model,
    tune_regression_model_gridsearch,
    get_tuning_results_dataframe,
    evaluate_optimized_model,
    compare_base_vs_optimized,
    save_tuning_results,
    save_optimized_metrics,
    save_base_vs_optimized_comparison,
    plot_base_vs_optimized_metric,
    run_regression_tuning_workflow,
)

from model_persistence import (
    save_model,
    load_model,
    save_final_metrics,
    create_final_summary_text,
    save_final_summary_text,
    verify_model_file,
    save_regression_final_artifacts,
)


## 3. Carga del dataset original

In [ ]:
df_raw = load_dataset(DATA_PATH)
df_raw.head()

## 4. Diagnostico inicial de calidad de datos

In [ ]:
print("Filas y columnas del dataset original:", df_raw.shape)

In [ ]:
df_raw.info()

In [ ]:
missing_report_raw = get_missing_values_report(df_raw)
display(missing_report_raw)

In [ ]:
duplicate_count = df_raw.duplicated().sum()
print("Filas duplicadas exactas:", duplicate_count)

In [ ]:
display(df_raw.describe(include="all").T)

## 5. Aplicacion del preprocesamiento base

En esta etapa se normalizan nombres de columnas, se limpian variables categoricas, se convierten variables numericas, se eliminan duplicados, se eliminan filas sin variable objetivo y se tratan outliers con IQR.

In [ ]:
df_clean = preprocess_regression_dataframe(df_raw)

print("Filas y columnas antes:", df_raw.shape)
print("Filas y columnas despues:", df_clean.shape)

df_clean.head()

## 6. Reporte posterior al preprocesamiento

In [ ]:
missing_report_clean = get_missing_values_report(df_clean)
display(missing_report_clean)

In [ ]:
duplicate_report = get_duplicate_report(df_raw, df_clean)
duplicate_report

In [ ]:
display(df_clean.describe(include="all").T)

## 7. Separacion de X e y despues del preprocesamiento

In [ ]:
X, y = split_features_target(df_clean)

print("Shape de X:", X.shape)
print("Shape de y:", y.shape)

display(X.head())
display(y.head())

## 8. Identificacion de variables numericas y categoricas

In [ ]:
print("Variables numericas:")
print(NUMERIC_FEATURES)

print("Variables categoricas:")
print(CATEGORICAL_FEATURES)

In [ ]:
display(df_clean[NUMERIC_FEATURES].dtypes)
display(df_clean[CATEGORICAL_FEATURES].dtypes)

## 9. Construccion del preprocesador de Scikit-learn

Las variables numericas se imputan por mediana y se escalan con StandardScaler. Las variables categoricas se imputan por moda y se transforman con OneHotEncoder.

In [ ]:
preprocessor = build_regression_preprocessor(
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

preprocessor

## 10. Guardado del dataset limpio

In [ ]:
save_processed_dataset(df_clean, PROCESSED_DATA_PATH)

print(f"Dataset limpio guardado en: {PROCESSED_DATA_PATH}")

## 11. Conclusion de la rama de preprocesamiento

En la rama de preprocesamiento se dejo preparado el dataset limpio, con validacion de columnas, limpieza de texto, conversion de numericos, eliminacion de duplicados, manejo de target faltante y tratamiento de outliers con IQR.

## 12. Entrenamiento base de modelos de regresion

En esta etapa se entrenan modelos base de regresion supervisada. El objetivo es preparar distintos algoritmos para que posteriormente puedan ser evaluados y comparados con metricas como MAE, RMSE y R2. En esta rama solo se realiza el entrenamiento inicial y la generacion de predicciones base.

In [ ]:
if PROCESSED_DATA_PATH.exists():
    df_model = pd.read_csv(PROCESSED_DATA_PATH)
else:
    df_model = df_clean.copy()

print("Dataset para entrenamiento:", df_model.shape)
df_model.head()

In [ ]:
X_train, X_test, y_train, y_test = split_regression_data(
    df=df_model,
    test_size=0.2,
    random_state=42,
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

In [ ]:
preprocessor = build_regression_preprocessor(
    numeric_features=NUMERIC_FEATURES,
    categorical_features=CATEGORICAL_FEATURES,
)

preprocessor

In [ ]:
regression_pipelines = build_regression_pipelines(
    preprocessor=preprocessor,
    random_state=42,
)

regression_pipelines.keys()

In [ ]:
trained_models = train_regression_models(
    models=regression_pipelines,
    X_train=X_train,
    y_train=y_train,
)

trained_models.keys()

In [ ]:
baseline_predictions = generate_baseline_predictions(
    trained_models=trained_models,
    X_test=X_test,
    y_test=y_test,
)

baseline_predictions.head()

In [ ]:
save_baseline_predictions(
    predictions_df=baseline_predictions,
    output_path=BASELINE_PREDICTIONS_PATH,
)

print(f"Predicciones base guardadas en: {BASELINE_PREDICTIONS_PATH}")

## 13. Resumen de modelos entrenados

In [ ]:
for model_name in trained_models:
    print(f"Modelo entrenado: {model_name}")

## 14. Conclusion de la rama de entrenamiento

En esta rama se implemento el entrenamiento base de modelos de regresion para la linea de trabajo de Cesar. Se cargo el dataset limpio, se separaron los datos en entrenamiento y prueba, se construyeron pipelines con preprocesamiento integrado y se entrenaron cuatro modelos: LinearRegression, DecisionTreeRegressor, RandomForestRegressor y GradientBoostingRegressor.

## 15. Evaluacion de modelos de regresion

En esta seccion se evaluan los modelos base entrenados anteriormente. Para modelos de regresion se utilizan metricas como MAE, RMSE y R2. MAE y RMSE miden el error del modelo en unidades de tiempo, mientras que R2 indica que proporcion de la variabilidad del tiempo de entrega es explicada por el modelo.

In [ ]:
metrics_df = evaluate_regression_models(
    trained_models=trained_models,
    X_test=X_test,
    y_test=y_test,
)

display(metrics_df)

## 16. Seleccion del mejor modelo base

In [ ]:
best_model_name = get_best_regression_model_name(metrics_df)
best_model = trained_models[best_model_name]

print("Mejor modelo base:", best_model_name)

best_model_metrics = metrics_df[metrics_df["model"] == best_model_name]
display(best_model_metrics)

El mejor modelo base se selecciona principalmente por el menor RMSE, ya que esta metrica penaliza con mayor fuerza los errores grandes. Tambien se revisan MAE y R2 para tener una interpretacion mas completa.

## 17. Predicciones del mejor modelo base

In [ ]:
best_predictions_df = create_predictions_dataframe(
    model=best_model,
    X_test=X_test,
    y_test=y_test,
)

display(best_predictions_df.head())

## 18. Guardado de metricas y resumen del mejor modelo

In [ ]:
save_metrics(metrics_df, METRICS_OUTPUT_PATH)
save_best_model_summary(metrics_df, BEST_MODEL_SUMMARY_PATH)

print(f"Metricas guardadas en: {METRICS_OUTPUT_PATH}")
print(f"Resumen del mejor modelo guardado en: {BEST_MODEL_SUMMARY_PATH}")

## 19. Visualizaciones de evaluacion

In [ ]:
plot_metric_comparison(metrics_df, "RMSE", RMSE_PLOT_PATH)
plot_metric_comparison(metrics_df, "MAE", MAE_PLOT_PATH)
plot_real_vs_predicted(best_predictions_df, REAL_VS_PRED_PLOT_PATH)
plot_residuals(best_predictions_df, RESIDUALS_PLOT_PATH)

print("Graficos guardados correctamente.")

## 20. Interpretacion de resultados

In [ ]:
best_row = metrics_df.sort_values(by=["RMSE", "MAE", "R2"], ascending=[True, True, False]).iloc[0]
best_name = best_row["model"]
best_mae = best_row["MAE"]
best_rmse = best_row["RMSE"]
best_r2 = best_row["R2"]

print(
    f"Segun la comparacion de metricas, el mejor modelo base fue {best_name}. "
    f"Este modelo obtuvo un MAE de {best_mae:.3f}, un RMSE de {best_rmse:.3f} y un R2 de {best_r2:.3f}."
)
print(
    f"El MAE indica que, en promedio, las predicciones se alejan aproximadamente {best_mae:.3f} minutos del tiempo real. "
    f"El RMSE penaliza mas los errores grandes y el R2 resume la variabilidad explicada por el modelo."
)
print("Estos resultados seran la linea base para la siguiente rama de tuning.")

## 21. Conclusion de la rama de evaluacion

En esta rama se evaluaron los modelos base de regresion entrenados previamente. Se calcularon metricas MAE, RMSE y R2 para comparar el rendimiento de LinearRegression, DecisionTreeRegressor, RandomForestRegressor y GradientBoostingRegressor. Tambien se generaron visualizaciones para comparar errores y analizar el comportamiento del mejor modelo base. La siguiente etapa sera optimizar hiperparametros del modelo con mejor rendimiento.

## 22. Optimizacion de hiperparametros

En esta seccion se optimiza el mejor modelo base usando GridSearchCV. La optimizacion busca encontrar una mejor combinacion de hiperparametros para reducir el error del modelo, principalmente el RMSE.

In [ ]:
best_model_name = get_best_regression_model_name(metrics_df)
print("Modelo base seleccionado para optimizacion:", best_model_name)

param_grid = get_param_grid_for_model(best_model_name)
param_grid

Los hiperparametros se definen con el prefijo `model__` porque el modelo se encuentra dentro de un `Pipeline` de Scikit-learn.

In [ ]:
tuning_results = run_regression_tuning_workflow(
    best_model_name=best_model_name,
    trained_models=trained_models,
    base_metrics_df=metrics_df,
    X_train=X_train,
    X_test=X_test,
    y_train=y_train,
    y_test=y_test,
    tuning_results_path=TUNING_RESULTS_PATH,
    optimized_metrics_path=OPTIMIZED_METRICS_PATH,
    comparison_path=BASE_VS_OPTIMIZED_PATH,
    rmse_plot_path=BASE_VS_OPT_RMSE_PLOT_PATH,
    mae_plot_path=BASE_VS_OPT_MAE_PLOT_PATH,
    optimized_real_vs_pred_path=OPTIMIZED_REAL_VS_PRED_PATH,
    optimized_residuals_path=OPTIMIZED_RESIDUALS_PATH,
)

## 23. Mejores hiperparametros encontrados

In [ ]:
print("Mejores hiperparametros:")
print(tuning_results["best_params"])

print("Mejor score de validacion cruzada:")
print(tuning_results["best_cv_score"])

## 24. Metricas del modelo optimizado

In [ ]:
optimized_metrics_df = tuning_results["optimized_metrics"]
display(optimized_metrics_df)

## 25. Comparacion modelo base vs modelo optimizado

In [ ]:
base_vs_optimized_df = tuning_results["comparison"]
display(base_vs_optimized_df)

Si el RMSE y el MAE disminuyen despues de la optimizacion, se puede afirmar que el ajuste de hiperparametros mejoro el rendimiento. Si las metricas empeoran o se mantienen iguales, se debe explicar que la configuracion base ya era suficientemente buena o que el espacio de busqueda no produjo una mejora.

## 26. Resultados de busqueda de hiperparametros

In [ ]:
tuning_results_df = tuning_results["tuning_results"]
display(tuning_results_df.head())

## 27. Interpretacion del impacto de la optimizacion

El modelo base seleccionado para optimizacion fue `NOMBRE_MODELO`. Despues de aplicar GridSearchCV, se obtuvo una version optimizada con nuevos hiperparametros. La comparacion entre el modelo base y el optimizado permite analizar si hubo mejora en MAE, RMSE y R2. Este analisis es importante porque la evaluacion solicita justificar el impacto de la optimizacion y no solo aplicar la tecnica de forma automatica.

In [ ]:
base_row = base_vs_optimized_df[base_vs_optimized_df["version"] == "base"].iloc[0]
opt_row = base_vs_optimized_df[base_vs_optimized_df["version"] == "optimized"].iloc[0]

print(
    f"Modelo base: {base_row['model']} | MAE={base_row['MAE']:.4f} | RMSE={base_row['RMSE']:.4f} | R2={base_row['R2']:.4f}"
)
print(
    f"Modelo optimizado: {opt_row['model']} | MAE={opt_row['MAE']:.4f} | RMSE={opt_row['RMSE']:.4f} | R2={opt_row['R2']:.4f}"
)

rmse_delta = base_row['RMSE'] - opt_row['RMSE']
mae_delta = base_row['MAE'] - opt_row['MAE']
r2_delta = opt_row['R2'] - base_row['R2']

print(f"Cambio RMSE (base - optimizado): {rmse_delta:.4f}")
print(f"Cambio MAE (base - optimizado): {mae_delta:.4f}")
print(f"Cambio R2 (optimizado - base): {r2_delta:.4f}")

## 28. Conclusion de la rama de optimizacion

En esta rama se aplico optimizacion de hiperparametros al mejor modelo base de regresion. Se utilizo GridSearchCV con validacion cruzada para buscar una combinacion de parametros que redujera el error del modelo. Luego se comparo el rendimiento del modelo base contra el optimizado usando MAE, RMSE y R2. Los resultados de esta rama permiten definir el modelo candidato final, que sera guardado en la siguiente etapa.

## 29. Seleccion del modelo final

En esta seccion se selecciona el modelo final de la linea de regresion de Cesar. Si la version optimizada mejora o mantiene un rendimiento competitivo frente al modelo base, se utiliza como modelo final. La decision se justifica considerando principalmente RMSE, y de forma complementaria MAE y R2.

In [ ]:
final_model = tuning_results["optimized_model"]
final_model_name = tuning_results["optimized_model_name"]
final_metrics_df = tuning_results["optimized_metrics"]

print("Modelo final seleccionado:", final_model_name)
display(final_metrics_df)

## 30. Guardado del modelo final y artefactos

In [ ]:
final_artifacts = save_regression_final_artifacts(
    final_model=final_model,
    final_model_name=final_model_name,
    final_metrics_df=final_metrics_df,
    model_output_path=FINAL_MODEL_PATH,
    final_metrics_output_path=FINAL_METRICS_PATH,
    final_summary_output_path=FINAL_SUMMARY_PATH,
    base_vs_optimized_df=base_vs_optimized_df,
)

final_artifacts

## 31. Verificacion de carga del modelo guardado

In [ ]:
loaded_model = load_model(FINAL_MODEL_PATH)

sample_predictions = loaded_model.predict(X_test.head())

print("Predicciones de prueba con modelo cargado:")
print(sample_predictions)

## 32. Resumen final de metricas

In [ ]:
display(final_metrics_df)

print("Archivo de modelo existe:", verify_model_file(FINAL_MODEL_PATH))

## 33. Conclusion final de la linea de regresion

En esta linea de trabajo se desarrollo un modelo de regresion supervisada para predecir el tiempo de entrega dentro del contexto Logistica 4.0. El proceso incluyo carga de datos, preprocesamiento, entrenamiento de varios modelos, evaluacion comparativa, optimizacion de hiperparametros y guardado del modelo final.

El modelo final seleccionado fue `linear_regression_optimized`, con MAE aproximado de `11.6258`, RMSE aproximado de `14.8058` y R2 aproximado de `-0.0008`. Estas metricas permiten interpretar el error promedio en minutos, la presencia de errores grandes y la capacidad explicativa del modelo.

Desde una perspectiva de negocio, este modelo puede apoyar la planificacion logistica, permitiendo estimaciones mas realistas del tiempo de entrega. Como mejora futura, se recomienda probar el modelo con mas datos reales, analizar importancia de variables e integrar nuevas variables externas como eventos, cortes de ruta o condiciones operativas especiales.